In [1]:
from DrissionPage import ChromiumOptions
path = r'"C:\\Program Files\\Google\\Chrome\\Application\\chrome.exe"'#请改为你电脑内chrome可执行文件路径
ChromiumOptions().set_browser_path(path).save()

配置已保存到文件: d:\code\elliptic2\.conda\Lib\site-packages\DrissionPage\_configs\configs.ini
以后程序可自动从文件加载配置


'd:\\code\\elliptic2\\.conda\\Lib\\site-packages\\DrissionPage\\_configs\\configs.ini'

In [2]:
#导入自动化模块
from DrissionPage import ChromiumPage
#打开浏览器 ->实例化浏览器对象，变量dp接收浏览器对象
dp = ChromiumPage()
#访问网站
#dp.get('https://search.jd.com/Search?keyword=%E7%94%B5%E8%84%91%E8%87%AA%E8%90%A5&enc=utf-8&pvid=5ea30b3cf94e489489c616b767c26f12&themeColor=&from=home&spmTag=YTAyMTkuYjAwMjM1Ni5jMDAwMDcyMTAua2V5d29yZF9lbnRlciU0MDE3Nzc3OTA2NjUwMjElMjMxNzc3NjAyMTgzNDgyMTg0Nzc0MDk5NCUyMzE5MjYzNjUwNjU')

In [3]:
#监听数据
#dp.listen.start('api?appid=search-pc-java&t')
#访问网站
#dp.get('https://search.jd.com/Search?keyword=%E7%94%B5%E8%84%91%E8%87%AA%E8%90%A5&enc=utf-8&pvid=5ea30b3cf94e489489c616b767c26f12&themeColor=&from=home&spmTag=YTAyMTkuYjAwMjM1Ni5jMDAwMDcyMTAua2V5d29yZF9lbnRlciU0MDE3Nzc3OTA2NjUwMjElMjMxNzc3NjAyMTgzNDgyMTg0Nzc0MDk5NCUyMzE5MjYzNjUwNjU')
#等待数据包加载
#resp = dp.listen.wait()
#获取返回数据内容
#json_data = resp.response.body
#print(json_data)

In [4]:
import csv
# 创建文件
f = open('cpu自营2.csv',mode='w',encoding='utf-8',newline='')
#字典写入方法
csv_writer = csv.DictWriter(f, fieldnames=[
        '商品名称',
        '价格',
        '店铺名称',
        '颜色',
        '评论数',
        '图片',
        '店铺ID',
        '卖点',
        'skuId',
        '总销量',
        '商品ID'
])

In [5]:
import re
import time

# 注意：listen.wait 第一个位置参数是 count（包个数），不是秒数。
# wait(10) = 等够 10 个包且默认 timeout=None 会一直等；限时应写 wait(timeout=10)。
# 比 'api?appid=search-pc-java&t' 更宽，减少 URL 微差异导致漏听
LISTEN_MASK = 'search-pc-java'
# 列表页滚动抓包
SCROLL_STEPS = 12
SCROLL_DELTA = 550
SCROLL_PAUSE = 1.0
# 最多翻页数（会与页面「共 N 页」取较小值）
MAX_PAGES = 100
SEARCH_KEYWORD = 'cpu自营'


def _unwrap_packets(r):
    """listen.wait 在 fit_count=False 时可能返回单包、列表或 False。"""
    if r is None or r is False:
        return []
    if isinstance(r, (list, tuple)):
        return [x for x in r if x is not None and x is not False]
    return [r]


def _normalize_selling_point(sp):
    if sp is None:
        return ''
    if isinstance(sp, (list, tuple)):
        return ';'.join(str(x) for x in sp)
    return str(sp)


def _process_one_json(json_data, seen_skus=None):
    if not isinstance(json_data, dict) or 'abBuriedTagMap' not in json_data:
        return 0
    data = json_data.get('data')
    if isinstance(data, dict):
        ware_list = data.get('wareList') or []
    else:
        # 部分接口/异常包里 data 是 list 或其它类型，没有 wareList
        ware_list = []
    if not isinstance(ware_list, list):
        return 0
    n = 0
    for index in ware_list:
        if not isinstance(index, dict):
            continue
        sku = (str(index.get('skuId') or index.get('wareId') or '')).strip()
        if seen_skus is not None and sku:
            if sku in seen_skus:
                continue
            seen_skus.add(sku)
        title = index.get('wareName') or ''
        new_title = re.sub(r'<.*?>', '', title)
        dit = {
            '商品名称': new_title,
            '价格': index.get('jdPrice', ''),
            '店铺名称': index.get('shopName', ''),
            '颜色': index.get('color', ''),
            '评论数': index.get('commentFuzzy', ''),
            '图片': index.get('imageurl', ''),
            '店铺ID': index.get('shopId', ''),
            '卖点': _normalize_selling_point(index.get('sellingPoint')),
            'skuId': index.get('skuId', ''),
            '总销量': index.get('totalSales', ''),
            '商品ID': index.get('wareId', ''),
        }
        csv_writer.writerow(dit)
        print(dit)
        n += 1
    return n


def _parse_total_pages(dp):
    """从分页区域「共 N 页」解析总页数；失败则返回 None。"""
    try:
        el = dp.ele('css:div[class*="_pagination_total"]', timeout=3)
        m = re.search(r'共\s*(\d+)\s*页', el.text)
        if m:
            return int(m.group(1))
    except Exception:
        pass
    return None


def _collect_page_packets(dp):
    """当前列表页：首屏 + 分段下滑 + 排空监听队列。"""
    packets = []
    packets.extend(_unwrap_packets(dp.listen.wait(timeout=25, fit_count=False)))
    for _ in range(SCROLL_STEPS):
        dp.scroll.down(SCROLL_DELTA)
        time.sleep(SCROLL_PAUSE)
        packets.extend(_unwrap_packets(dp.listen.wait(timeout=2.5, fit_count=False)))
    time.sleep(1.5)
    miss = 0
    while miss < 4:
        chunk = _unwrap_packets(dp.listen.wait(timeout=4, fit_count=False))
        if not chunk:
            miss += 1
        else:
            packets.extend(chunk)
            miss = 0
    return packets


def _click_next_page(dp):
    """点击「下一页」（class 带 hash，用包含匹配）。"""
    btn = dp.ele('css:div[class*="_pagination_next"]', timeout=8)
    btn.click()


# 访问网站
dp.get('https://search.jd.com/Search?keyword=&enc=utf-8')
dp.ele('css:.jd_pc_search_bar_react_search_input').input(SEARCH_KEYWORD)
dp.listen.start(LISTEN_MASK)
dp.ele('css:.jd_pc_search_bar_react_search_btn').click()
time.sleep(1.5)

total_pg = _parse_total_pages(dp)
pages_to_do = min(MAX_PAGES, total_pg) if total_pg else MAX_PAGES
print('检测到总页数:', total_pg, '| 本次爬取页数:', pages_to_do)

seen_skus = set()
packet_total = 0
row_total = 0

for pi in range(1, pages_to_do + 1):
    print(f'===== 第 {pi}/{pages_to_do} 页 =====')
    packets = _collect_page_packets(dp)
    packet_total += len(packets)
    for resp in packets:
        try:
            body = resp.response.body
        except Exception:
            continue
        row_total += _process_one_json(body, seen_skus)
    if pi >= pages_to_do:
        break
    try:
        _click_next_page(dp)
        time.sleep(2.0)
    except Exception as ex:
        print('翻页结束（可能已是最后一页或选择器失效）:', ex)
        break

print(
    '数据包合计', packet_total,
    '，累计写入', row_total, '行（已按 skuId 去重；无 sku 的仍按条写入）',
)

{'商品名称': '华硕TUF GAMING B760M-PLUS WIFI D4 重炮手主板  CPU 14600KF/14700KF（Intel B760/LGA 1700）', '价格': '1219.00', '店铺名称': '华硕京东自营旗舰店', '颜色': '【重炮手WIFID4】B760M', '评论数': '10万+', '图片': 'jfs/t1/179419/35/31046/109862/63b7bca3F9902d146/d8e891b8795c2480.jpg', '店铺ID': '1000000225', '卖点': '15万+人加购;90万+人浏览;用料扎实;重炮手', 'skuId': '100041216184', '总销量': '10万+', '商品ID': '100041216184'}
{'商品名称': 'AMD锐龙 5 9600X处理器(R5) 4nm 6核12线程 加速频率至高5.4GHz 盒装CPU 畅玩打瓦/三角洲/CSGO', '价格': '1349.00', '店铺名称': 'AMD京东自营旗舰店', '颜色': '9600X盒装CPU', '评论数': '2万+', '图片': 'jfs/t1/424061/14/19093/95421/69f32c01Fabbcf815/008332032050563e.jpg', '店铺ID': '1000000706', '卖点': '超频发烧利器;优质畅玩;畅玩大作;高效游戏体验', 'skuId': '100110580225', '总销量': '5万+', '商品ID': '100110580225'}
{'商品名称': 'AMD锐龙5 5600处理器(r5)7nm 6核12线程 加速频率至高4.4GHz AM4盒装CPU 畅玩无畏契约/CSGO', '价格': '879.00', '店铺名称': 'AMD京东自营旗舰店', '颜色': 'R5-5600|3.5GHz|6核12线程', '评论数': '5万+', '图片': 'jfs/t1/425843/3/14133/96532/69f32c74Fd9624791/0083320320299be8.jpg', '店铺ID': '1000000706', '卖点': '快速编译;多线程支持;游戏强芯;高效能核芯',